# Notebook 00 — Setup y exploración general

**Objetivo:** Cargar los tres ficheros fuente, aplicar limpieza de missings del ESS,
calcular todas las variables derivadas y guardar versiones limpias para los notebooks siguientes.

**Outputs:**
- `data/processed/ESS11_clean.csv`
- `data/processed/ESS7_clean.csv`

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Rutas
ROOT = Path('../')
RAW  = ROOT / 'ESS11'
RAW7 = ROOT / 'ESS7'
EIGE = ROOT / 'EIGE-scores'
OUT  = ROOT / 'data' / 'processed'
OUT.mkdir(parents=True, exist_ok=True)

print('Rutas OK')

Rutas OK


## 1. Variables del proyecto y códigos de missing

In [2]:
# Códigos de missing en el ESS
ESS_MISSING = [
    6, 7, 8, 9,
    66, 77, 88, 99,
    666, 777, 888, 999,
    6666, 7777, 8888, 9999,
    66666, 77777, 88888, 99999
]

# Todas las variables de interés del proyecto
VARS_PROYECTO = [
    # Salud y bienestar
    'health', 'hlthhmp', 'happy', 'stflife', 'wrhpp',
    'fltdpr', 'fltlnl', 'enjlf', 'fltsd', 'slprl',
    # Género, trabajo e ingresos
    'gndr', 'agea', 'edulvlb',
    'pdwrk', 'mnactic', 'emplrel', 'wkhtot', 'wkhct', 'wkdcorga',
    'hswrk', 'chldhhe', 'hinctnta',
    # Módulo género (solo R11)
    'dscrgnd', 'eqwrkbg', 'eqpaybg', 'mascfel', 'femifel',
    # Acceso sanitario
    'hltprhc', 'hltphhc', 'dshltgp', 'dshltms', 'cnfpplh', 'stfhlth',
]

# Variables geográficas/temporales (no necesitan limpieza de missing numérico)
VARS_GEO = ['cntry', 'region', 'domicil', 'essround']
VARS_PESOS = ['dweight', 'pspwght', 'pweight', 'anweight']

def clean_ess_missing(df, columns):
    """Convierte códigos de missing del ESS a NaN."""
    df = df.copy()
    for col in columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            df[col] = df[col].replace(ESS_MISSING, np.nan)
    return df

print('Funciones y constantes definidas')

Funciones y constantes definidas


## 2. Cargar ESS R11

In [3]:
ess11_path = RAW / 'ESS11e04_1.csv'
print(f'Cargando {ess11_path} ...')

ess11_raw = pd.read_csv(ess11_path, low_memory=False)
print(f'ESS R11 cargado: {ess11_raw.shape[0]:,} filas × {ess11_raw.shape[1]} columnas')

Cargando ../ESS11/ESS11e04_1.csv ...


ESS R11 cargado: 50,116 filas × 691 columnas


In [4]:
# Seleccionar columnas relevantes (las que existen en el fichero)
cols_r11 = VARS_GEO + VARS_PESOS + [v for v in VARS_PROYECTO if v in ess11_raw.columns]
missing_r11 = [v for v in VARS_PROYECTO if v not in ess11_raw.columns]

print(f'Columnas seleccionadas: {len(cols_r11)}')
if missing_r11:
    print(f'Variables NO encontradas en R11: {missing_r11}')

ess11 = ess11_raw[cols_r11].copy()

# Limpiar missings
ess11 = clean_ess_missing(ess11, [v for v in VARS_PROYECTO if v in ess11.columns])
# cntry como string limpio
ess11['cntry'] = ess11['cntry'].str.strip()

print(f'\nESS R11 limpio: {ess11.shape}')
print(f'Países: {sorted(ess11["cntry"].unique())}')

Columnas seleccionadas: 41

ESS R11 limpio: (50116, 41)
Países: ['AT', 'BE', 'BG', 'CH', 'CY', 'DE', 'EE', 'ES', 'FI', 'FR', 'GB', 'GR', 'HR', 'HU', 'IE', 'IL', 'IS', 'IT', 'LT', 'LV', 'ME', 'NL', 'NO', 'PL', 'PT', 'RS', 'SE', 'SI', 'SK', 'UA']


## 3. Cargar ESS R7

In [5]:
ess7_path = RAW7 / 'ESS7e02_3.csv'
print(f'Cargando {ess7_path} ...')

ess7_raw = pd.read_csv(ess7_path, low_memory=False)
print(f'ESS R7 cargado: {ess7_raw.shape[0]:,} filas × {ess7_raw.shape[1]} columnas')

Cargando ../ESS7/ESS7e02_3.csv ...


ESS R7 cargado: 40,185 filas × 602 columnas


In [6]:
# Variables disponibles en R7 (el módulo género es solo R11)
VARS_SOLO_R11 = ['dscrgnd', 'eqwrkbg', 'eqpaybg', 'mascfel', 'femifel',
                 'hltprhc', 'hltphhc', 'dshltgp', 'dshltms', 'cnfpplh']
VARS_R7 = [v for v in VARS_PROYECTO if v not in VARS_SOLO_R11]

cols_r7 = VARS_GEO + VARS_PESOS + [v for v in VARS_R7 if v in ess7_raw.columns]
missing_r7 = [v for v in VARS_R7 if v not in ess7_raw.columns]

print(f'Columnas seleccionadas: {len(cols_r7)}')
if missing_r7:
    print(f'Variables NO encontradas en R7: {missing_r7}')

ess7 = ess7_raw[cols_r7].copy()
ess7 = clean_ess_missing(ess7, [v for v in VARS_R7 if v in ess7.columns])
ess7['cntry'] = ess7['cntry'].str.strip()

print(f'\nESS R7 limpio: {ess7.shape}')
print(f'Países: {sorted(ess7["cntry"].unique())}')

Columnas seleccionadas: 31

ESS R7 limpio: (40185, 31)
Países: ['AT', 'BE', 'CH', 'CZ', 'DE', 'DK', 'EE', 'ES', 'FI', 'FR', 'GB', 'HU', 'IE', 'IL', 'LT', 'NL', 'NO', 'PL', 'PT', 'SE', 'SI']


## 4. Verificar presencia de España

In [7]:
for nombre, df in [('R11', ess11), ('R7', ess7)]:
    es = df[df['cntry'] == 'ES']
    assert len(es) > 0, f'España NO encontrada en {nombre}!'
    gndr_counts = es['gndr'].value_counts().sort_index()
    print(f'España en {nombre}: {len(es):,} registros')
    print(f'  Hombres (1): {gndr_counts.get(1, 0):,}  |  Mujeres (2): {gndr_counts.get(2, 0):,}')
    print()

España en R11: 1,844 registros
  Hombres (1): 875  |  Mujeres (2): 969

España en R7: 1,925 registros
  Hombres (1): 988  |  Mujeres (2): 937



## 5. Variables derivadas

In [8]:
def calcular_idx_wellbeing(df):
    """
    Índice de bienestar mental compuesto (0–10, mayor = mejor bienestar).
    Componentes: happy, stflife, wrhpp, enjlf,
                 health_inv (1→muy buena, 5→muy mala, invertida y reescalada),
                 fltdpr_inv, fltlnl_inv, fltsd_inv (0=nunca, 10=siempre → invertidas).
    """
    df = df.copy()
    df['health_inv'] = (5 - df['health']) / 4 * 10  # 1→10, 5→0
    df['fltdpr_inv'] = 10 - df['fltdpr']
    df['fltlnl_inv'] = 10 - df['fltlnl']
    df['fltsd_inv']  = 10 - df['fltsd']

    componentes = ['happy', 'stflife', 'wrhpp', 'enjlf',
                   'health_inv', 'fltdpr_inv', 'fltlnl_inv', 'fltsd_inv']
    # Solo calcular si al menos la mitad de componentes están disponibles
    disponibles = [c for c in componentes if c in df.columns]
    df['IDX_WELLBEING'] = df[disponibles].mean(axis=1, skipna=True)
    # Marcar como NaN si más de la mitad son NaN
    n_nan = df[disponibles].isnull().sum(axis=1)
    df.loc[n_nan > len(disponibles) / 2, 'IDX_WELLBEING'] = np.nan
    return df


def calcular_brecha_genero(df):
    """
    Diferencia de IDX_WELLBEING medio entre hombres y mujeres por país.
    Valor positivo = hombres con mayor bienestar que mujeres.
    """
    media = (
        df.groupby(['cntry', 'gndr'])['IDX_WELLBEING']
        .mean()
        .unstack('gndr')
        .rename(columns={1: 'wellbeing_hombres', 2: 'wellbeing_mujeres'})
    )
    media['BRECHA_GNDR'] = media['wellbeing_hombres'] - media['wellbeing_mujeres']
    return media.reset_index()


def calcular_carga_total(df):
    """
    Proxy de carga laboral total = horas remuneradas + proxy de cuidados.
    Proxy cuidados: mujer + hijos en casa + (trabajo doméstico o no trabaja) → +20h.
    """
    df = df.copy()
    df['proxy_cuidados'] = 0
    if all(c in df.columns for c in ['gndr', 'chldhhe', 'hswrk', 'pdwrk']):
        mascara = (
            (df['gndr'] == 2) &
            (df['chldhhe'] > 0) &
            ((df['hswrk'] == 1) | (df['pdwrk'] == 2))
        )
        df.loc[mascara, 'proxy_cuidados'] = 20
    df['CARGA_TOTAL'] = df['wkhtot'].fillna(0) + df['proxy_cuidados']
    return df


def calcular_delta_temporal(df_r7, df_r11):
    """
    Diferencia de IDX_WELLBEING medio entre R7 (2014) y R11 (2024) por país y género.
    Valor positivo = mejora del bienestar.
    """
    media_r7 = (
        df_r7.groupby(['cntry', 'gndr'])['IDX_WELLBEING']
        .mean().reset_index().rename(columns={'IDX_WELLBEING': 'wb_r7'})
    )
    media_r11 = (
        df_r11.groupby(['cntry', 'gndr'])['IDX_WELLBEING']
        .mean().reset_index().rename(columns={'IDX_WELLBEING': 'wb_r11'})
    )
    delta = media_r7.merge(media_r11, on=['cntry', 'gndr'], how='inner')
    delta['DELTA_R7_R11'] = delta['wb_r11'] - delta['wb_r7']
    return delta


print('Funciones de variables derivadas definidas')

Funciones de variables derivadas definidas


## 6. Calcular variables derivadas en ambos datasets

In [9]:
# IDX_WELLBEING y CARGA_TOTAL en R11
ess11 = calcular_idx_wellbeing(ess11)
ess11 = calcular_carga_total(ess11)

print('ESS R11 — IDX_WELLBEING:')
print(f'  No nulos: {ess11["IDX_WELLBEING"].notna().sum():,} / {len(ess11):,}')
print(f'  Media global: {ess11["IDX_WELLBEING"].mean():.3f}')
print(f'  Por género:')
print(ess11.groupby('gndr')['IDX_WELLBEING'].agg(['mean', 'count']).round(3))

ESS R11 — IDX_WELLBEING:
  No nulos: 50,003 / 50,116
  Media global: 6.361
  Por género:
       mean  count
gndr              
1     6.428  23037
2     6.304  26966


In [10]:
# IDX_WELLBEING y CARGA_TOTAL en R7
ess7 = calcular_idx_wellbeing(ess7)
ess7 = calcular_carga_total(ess7)

print('ESS R7 — IDX_WELLBEING:')
print(f'  No nulos: {ess7["IDX_WELLBEING"].notna().sum():,} / {len(ess7):,}')
print(f'  Media global: {ess7["IDX_WELLBEING"].mean():.3f}')
print(f'  Por género:')
print(ess7.groupby('gndr')['IDX_WELLBEING'].agg(['mean', 'count']).round(3))

ESS R7 — IDX_WELLBEING:
  No nulos: 40,069 / 40,185
  Media global: 6.427
  Por género:
       mean  count
gndr              
1.0   6.500  18821
2.0   6.362  21226


## 7. BRECHA_GNDR por país (R11)

In [11]:
brecha_r11 = calcular_brecha_genero(ess11)
print('Brecha de bienestar por país (R11) — top 10 mayor brecha:')
print(brecha_r11.sort_values('BRECHA_GNDR', ascending=False).head(10).to_string(index=False))

Brecha de bienestar por país (R11) — top 10 mayor brecha:
cntry  wellbeing_hombres  wellbeing_mujeres  BRECHA_GNDR
   PT           6.312312           6.031083     0.281228
   CY           6.490345           6.212965     0.277380
   ES           6.561997           6.356964     0.205032
   SK           6.232290           6.052322     0.179968
   IT           6.337323           6.163406     0.173917
   BE           6.628617           6.458541     0.170076
   FR           6.522282           6.355024     0.167258
   RS           6.329271           6.164918     0.164352
   UA           5.810594           5.680478     0.130116
   GR           6.383765           6.257688     0.126077


In [12]:
# España específicamente
es_brecha = brecha_r11[brecha_r11['cntry'] == 'ES']
print('España (R11):')
print(es_brecha.to_string(index=False))

España (R11):
cntry  wellbeing_hombres  wellbeing_mujeres  BRECHA_GNDR
   ES           6.561997           6.356964     0.205032


## 8. DELTA_R7_R11

In [13]:
delta = calcular_delta_temporal(ess7, ess11)
print(f'Países presentes en ambas rondas: {delta["cntry"].nunique()}')
print('\nDelta por género — media global:')
print(delta.groupby('gndr')[['wb_r7', 'wb_r11', 'DELTA_R7_R11']].mean().round(3))

print('\nEspaña:')
print(delta[delta['cntry'] == 'ES'].to_string(index=False))

Países presentes en ambas rondas: 19

Delta por género — media global:
      wb_r7  wb_r11  DELTA_R7_R11
gndr                             
1.0   6.482   6.524         0.042
2.0   6.364   6.435         0.071

España:
cntry  gndr    wb_r7   wb_r11  DELTA_R7_R11
   ES   1.0 6.418576 6.561997      0.143420
   ES   2.0 6.229244 6.356964      0.127721


## 9. Conteo de registros por país y género

In [14]:
print('=== ESS R11: registros por país y género ===')
tabla_r11 = (
    ess11.groupby(['cntry', 'gndr'])
    .size()
    .unstack('gndr')
    .rename(columns={1: 'Hombres', 2: 'Mujeres'})
)
tabla_r11['Total'] = tabla_r11.sum(axis=1)
print(tabla_r11.to_string())
print(f'\nTotal R11: {tabla_r11["Total"].sum():,}')

=== ESS R11: registros por país y género ===
gndr   Hombres  Mujeres  Total
cntry                         
AT         993     1361   2354
BE         809      785   1594
BG        1064     1175   2239
CH         697      687   1384
CY         309      376    685
DE        1214     1206   2420
EE         577      716   1293
ES         875      969   1844
FI         770      793   1563
FR         874      897   1771
GB         824      860   1684
GR        1239     1518   2757
HR         711      852   1563
HU         835     1283   2118
IE         906     1111   2017
IL         440      466    906
IS         418      424    842
IT        1339     1526   2865
LT         526      839   1365
LV         436      816   1252
ME         867      742   1609
NL         843      852   1695
NO         673      664   1337
PL         675      767   1442
PT         578      795   1373
RS         731      832   1563
SE         642      588   1230
SI         608      640   1248
SK         671      771  

In [15]:
print('=== ESS R7: registros por país y género ===')
tabla_r7 = (
    ess7.groupby(['cntry', 'gndr'])
    .size()
    .unstack('gndr')
    .rename(columns={1: 'Hombres', 2: 'Mujeres'})
)
tabla_r7['Total'] = tabla_r7.sum(axis=1)
print(tabla_r7.to_string())
print(f'\nTotal R7: {tabla_r7["Total"].sum():,}')

=== ESS R7: registros por país y género ===
gndr   Hombres  Mujeres  Total
cntry                         
AT         853      942   1795
BE         896      873   1769
CH         766      766   1532
CZ         998     1128   2126
DE        1545     1500   3045
DK         779      723   1502
EE         835     1216   2051
ES         988      937   1925
FI        1027     1060   2087
FR         913     1004   1917
GB        1024     1240   2264
HU         722      976   1698
IE        1102     1288   2390
IL        1164     1398   2562
LT         869     1381   2250
NL         859     1060   1919
NO         764      672   1436
PL         740      875   1615
PT         571      694   1265
SE         893      898   1791
SI         563      661   1224

Total R7: 40,163


## 10. Calidad de las variables clave

In [16]:
vars_check = ['IDX_WELLBEING', 'health', 'happy', 'stflife', 'wrhpp',
              'fltdpr', 'fltlnl', 'fltsd', 'enjlf',
              'gndr', 'wkhtot', 'hinctnta', 'CARGA_TOTAL']

print('=== Calidad variables R11 ===')
for v in vars_check:
    if v in ess11.columns:
        n_total = len(ess11)
        n_validos = ess11[v].notna().sum()
        pct = n_validos / n_total * 100
        print(f'{v:<20} válidos: {n_validos:>6,} / {n_total:,}  ({pct:.1f}%)')

=== Calidad variables R11 ===
IDX_WELLBEING        válidos: 50,003 / 50,116  (99.8%)
health               válidos: 50,040 / 50,116  (99.8%)
happy                válidos: 14,038 / 50,116  (28.0%)
stflife              válidos: 16,256 / 50,116  (32.4%)
wrhpp                válidos: 49,679 / 50,116  (99.1%)


fltdpr               válidos: 49,834 / 50,116  (99.4%)
fltlnl               válidos: 49,829 / 50,116  (99.4%)
fltsd                válidos: 49,835 / 50,116  (99.4%)
enjlf                válidos: 49,693 / 50,116  (99.2%)
gndr                 válidos: 50,116 / 50,116  (100.0%)
wkhtot               válidos: 41,923 / 50,116  (83.7%)
hinctnta             válidos: 23,554 / 50,116  (47.0%)
CARGA_TOTAL          válidos: 50,116 / 50,116  (100.0%)


## 11. Guardar ficheros limpios

In [17]:
def verificar_csv_flourish(df, nombre):
    print(f'\n=== {nombre} ===')
    print(f'Filas: {len(df)}')
    print(f'Columnas: {list(df.columns)}')
    nulos = df.isnull().sum()
    nulos_con = nulos[nulos > 0]
    if len(nulos_con):
        print(f'Valores nulos (solo columnas afectadas):\n{nulos_con}')
    else:
        print('Sin valores nulos')
    print(f'Muestra:\n{df.head(3).to_string()}')
    assert not df.isin([float('inf'), float('-inf')]).any().any(), 'Hay infinitos'
    assert not df.isnull().all(axis=1).any(), 'Hay filas completamente nulas'
    print('✓ OK')


# Guardar R11
out11 = OUT / 'ESS11_clean.csv'
ess11.to_csv(out11, index=False)
print(f'Guardado: {out11}  ({len(ess11):,} filas)')

# Guardar R7
out7 = OUT / 'ESS7_clean.csv'
ess7.to_csv(out7, index=False)
print(f'Guardado: {out7}  ({len(ess7):,} filas)')

Guardado: ../data/processed/ESS11_clean.csv  (50,116 filas)


Guardado: ../data/processed/ESS7_clean.csv  (40,185 filas)


In [18]:
# Verificar los ficheros guardados
ess11_check = pd.read_csv(OUT / 'ESS11_clean.csv')
ess7_check  = pd.read_csv(OUT / 'ESS7_clean.csv')

verificar_csv_flourish(ess11_check[['cntry', 'gndr', 'IDX_WELLBEING', 'CARGA_TOTAL', 'BRECHA_GNDR' if 'BRECHA_GNDR' in ess11_check.columns else 'IDX_WELLBEING']], 'ESS11_clean (muestra)')
verificar_csv_flourish(ess7_check[['cntry', 'gndr', 'IDX_WELLBEING', 'CARGA_TOTAL']], 'ESS7_clean (muestra)')


=== ESS11_clean (muestra) ===
Filas: 50116
Columnas: ['cntry', 'gndr', 'IDX_WELLBEING', 'CARGA_TOTAL', 'IDX_WELLBEING']
Valores nulos (solo columnas afectadas):
IDX_WELLBEING    113
IDX_WELLBEING    113
dtype: int64
Muestra:
  cntry  gndr  IDX_WELLBEING  CARGA_TOTAL  IDX_WELLBEING
0    AT     1       6.333333          0.0       6.333333
1    AT     2       6.250000         32.0       6.250000
2    AT     2       7.428571         25.0       7.428571
✓ OK

=== ESS7_clean (muestra) ===
Filas: 40185
Columnas: ['cntry', 'gndr', 'IDX_WELLBEING', 'CARGA_TOTAL']
Valores nulos (solo columnas afectadas):
gndr              22
IDX_WELLBEING    116
dtype: int64
Muestra:
  cntry  gndr  IDX_WELLBEING  CARGA_TOTAL
0    AT   1.0       6.928571         44.0
1    AT   1.0       5.714286         40.0
2    AT   2.0       6.714286         20.0
✓ OK


## 12. Resumen final

In [19]:
print('=' * 60)
print('RESUMEN NOTEBOOK 00')
print('=' * 60)
print(f'ESS R11: {len(ess11):,} registros, {ess11["cntry"].nunique()} países')
print(f'ESS R7:  {len(ess7):,} registros, {ess7["cntry"].nunique()} países')
print(f'Países en ambas rondas: {delta["cntry"].nunique()}')
print()
print('IDX_WELLBEING R11 por género:')
wb_gndr = ess11.groupby('gndr')['IDX_WELLBEING'].mean().round(3)
print(f'  Hombres (1): {wb_gndr.get(1, "N/A")}')
print(f'  Mujeres (2): {wb_gndr.get(2, "N/A")}')
print(f'  Brecha media: {(wb_gndr.get(1, 0) - wb_gndr.get(2, 0)):.3f}')
print()
print('Ficheros generados:')
print(f'  → {out11}')
print(f'  → {out7}')
print()
print('Próximos pasos:')
print('  Ejecutar notebooks 01–05 en cualquier orden')

RESUMEN NOTEBOOK 00
ESS R11: 50,116 registros, 30 países
ESS R7:  40,185 registros, 21 países
Países en ambas rondas: 19

IDX_WELLBEING R11 por género:
  Hombres (1): 6.428
  Mujeres (2): 6.304
  Brecha media: 0.124

Ficheros generados:
  → ../data/processed/ESS11_clean.csv
  → ../data/processed/ESS7_clean.csv

Próximos pasos:
  Ejecutar notebooks 01–05 en cualquier orden
